# colab_12 — RNA velocity demo (scVelo, dentate gyrus)

## Why this is a demo, not a project notebook

RNA velocity (scVelo) needs **spliced + unspliced count matrices** per cell, generated by running velocyto on aligned BAM files. Our actual datasets don't have these:
- **Bhaduri 2020** — the GEO-archived expression matrix is `cellranger aggr --normalize=mapped` output (Session 23 colab_07b §1b verified). No BAMs available without re-running cellranger from SRA fastqs (multi-day compute).
- **Bhaduri 2021** — NeMO ships filtered count tarballs (`GRCh38.mex.tar.gz`), not BAMs or velocyto loom files.

So scVelo is fundamentally not runnable on `integrated_annotated_100k.h5ad`. This notebook instead runs the full scVelo pipeline on `scv.datasets.dentategyrus()` — a public mouse dentate gyrus dataset (Hochgerner et al. 2018) bundled with scVelo, with proper spliced/unspliced matrices. It is thematically close to our project (hippocampal neurogenesis: radial-glia-like cells → neuroblasts → immature granule cells → mature granule cells, the same kind of progenitor-to-neuron lineage we'd be analysing in organoids).

The point is to see *what scVelo's outputs look like when the data supports it* — what plots a successful velocity analysis produces and how they should be read.

## What this notebook produces

1. **Velocity stream plot** — arrows showing the predicted near-future state of each cell projected onto UMAP. The headline scVelo plot.
2. **Velocity confidence** — per-cell consistency of the velocity field with its neighbours. Low values = noisy / direction unreliable.
3. **Velocity pseudotime + latent time** — two ordering schemes derived from velocity, not from diffusion distances. Latent time (dynamical model only) is the most biologically interpretable.
4. **Top dynamical genes** — genes that carry the most velocity signal. Phase portraits (spliced vs unspliced with model fits) show the splicing dynamics underlying velocity for each gene.
5. **Heatmap of dynamical genes ordered by latent time** — gene-expression cascade along the trajectory.
6. **Directed PAGA** — PAGA graph with edge directions inferred from velocity.

## Mode

Uses scVelo's **dynamical mode** (`scv.tl.recover_dynamics` + `mode='dynamical'`). Slower than the default stochastic mode (ca. 5–10 min on this dataset) but enables `latent_time` and proper `rank_dynamical_genes`. This is the recommended mode for figures.

## 0. Setup

### 0a — Install dependencies, import packages

Installs `scvelo` (pulls scanpy, anndata, numba, loompy as deps). No Drive mount needed — the demo dataset is downloaded by scVelo on first call into `data/` in the working directory.

In [ ]:
!pip install -q scvelo

import numpy as np
import pandas as pd
import scvelo as scv
import scanpy as sc
import matplotlib.pyplot as plt

scv.set_figure_params('scvelo', dpi=80, frameon=False)
scv.settings.verbosity = 3
scv.settings.presenter_view = True

print('scvelo', scv.__version__, '| scanpy', sc.__version__)

## 1. Load the dentate gyrus demo dataset

### 1a — `scv.datasets.dentategyrus()`

Loads the bundled mouse dentate gyrus AnnData (Hochgerner et al. 2018). Has `layers['spliced']` and `layers['unspliced']` already populated (these are the velocyto outputs that we don't have for our project). Cell types are pre-annotated as `clusters` in obs — Granule mature, Granule immature, Neuroblast, OPC, Radial Glia-like, OL, Astrocytes, Endothelial, Cajal Retzius, Microglia.

In [ ]:
adata = scv.datasets.dentategyrus()
print(adata)
print()
print('Cell types:')
print(adata.obs['clusters'].value_counts().to_string())
print()
print('Layers (spliced/unspliced are the velocyto outputs):')
for k, v in adata.layers.items():
    print(f'  {k:12} shape={v.shape} dtype={v.dtype}')

## 2. Preprocessing

### 2a — Filter, normalize, log-transform

`scv.pp.filter_and_normalize` is scVelo's all-in-one preprocessing helper. It:
1. Filters out genes with low spliced/unspliced counts (`min_shared_counts=20`).
2. Selects HVGs (`n_top_genes=2000`).
3. Normalizes each layer by total counts and log-transforms.

Applied to all layers (spliced, unspliced, X) so velocity later sees consistent normalization.

In [ ]:
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
print(f'Shape after filter+normalize: {adata.shape}')

### 2b — Compute first/second-order moments

`scv.pp.moments` builds a kNN graph in PCA space and computes per-cell first (mean) and second (variance) moments of spliced and unspliced counts over the neighbourhood. These moments smooth out single-cell sparsity and are the inputs to the velocity model in §3.

`n_pcs=30` and `n_neighbors=30` are scVelo defaults — same order of magnitude as our scanpy pipeline (we used 30 PCs and 15 neighbours).

In [ ]:
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
print(f'Moments computed; layers now include: {list(adata.layers.keys())}')

## 3. Dynamical model — recover transcriptional dynamics

### 3a — `scv.tl.recover_dynamics`

Fits a per-gene splicing kinetics model: each gene gets transcription, splicing, and degradation rates plus per-cell latent times under the model. This is the slow step (ca. 3–7 min on this dataset), but it enables the dynamical velocity mode and `latent_time`. Without this step, you can still run velocity in stochastic / deterministic mode but lose latent time and the dynamical-genes ranking.

In [ ]:
scv.tl.recover_dynamics(adata, n_jobs=4)
print('Recovered dynamics for', adata.var['fit_likelihood'].notna().sum(), 'genes.')

## 4. Compute velocity

### 4a — `scv.tl.velocity` (mode='dynamical')

At this point the gene-specific kinetics from §3 are combined with the per-cell unspliced/spliced moments to compute a velocity vector for each cell in gene space. The velocity for a gene at a cell is the time-derivative of the spliced count under the fitted model — positive = gene is being induced, negative = gene is being repressed.

In [ ]:
scv.tl.velocity(adata, mode='dynamical')
print(f'Velocity computed; new layer: {[k for k in adata.layers if "velocity" in k]}')

### 4b — Velocity graph

`scv.tl.velocity_graph` projects the velocity field onto a cell-cell transition graph. For each cell, computes cosine similarity between its velocity vector and the displacement to each of its kNN neighbours — the result is a (cells × cells) sparse matrix where high values = strong predicted transition. Used by the §5 stream and arrow plots to project velocity onto 2D embeddings.

In [ ]:
scv.tl.velocity_graph(adata, n_jobs=4)
print('Velocity graph computed.')

## 5. Velocity visualizations

### 5a — Stream plot on UMAP, coloured by cell type

The canonical scVelo figure. Streamlines integrate the projected velocity field; cells coloured by type. Read it as: starting from any cell, follow the stream to see where the model predicts that cell will move next.

For dentate gyrus the expected pattern: streamlines flow from Radial Glia-like → Neuroblast → Granule immature → Granule mature, with side branches into OPC/OL and astrocyte fates. If the streams instead flow inward to a single point or have opposing eddies, velocity is unreliable for that region.

In [ ]:
scv.pl.velocity_embedding_stream(
    adata, basis='umap', color='clusters',
    legend_loc='right margin',
    save='velocity_stream_clusters.png',
)

### 5b — Per-cell arrows on a UMAP grid

Same velocity field, but instead of streamlines, each grid cell gets an averaged arrow. Useful for spotting localized direction changes that streamlines smooth over.

In [ ]:
scv.pl.velocity_embedding_grid(
    adata, basis='umap', color='clusters',
    arrow_length=2, arrow_size=1.5,
    legend_loc='right margin',
    save='velocity_grid_clusters.png',
)

### 5c — Velocity confidence and length per cell

Two per-cell velocity QC scores:
- **velocity_length** — magnitude of the velocity vector. Higher = cell is changing state faster.
- **velocity_confidence** — coherence of this cell's velocity with its neighbours' velocities. Higher = direction is consistent across the local neighbourhood; low values flag cells where velocity is noisy / not interpretable.

Look for low-confidence regions on UMAP — they correspond to areas where the velocity arrows can't be trusted.

In [ ]:
scv.tl.velocity_confidence(adata)
scv.pl.scatter(
    adata, c=('velocity_length', 'velocity_confidence'),
    cmap='coolwarm', perc=[5, 95],
    save='velocity_confidence.png',
)

## 6. Velocity pseudotime and latent time

### 6a — Velocity pseudotime

`scv.tl.velocity_pseudotime` orders cells along the velocity-graph random-walk distance from automatically detected root cells. Conceptually similar to DPT but using the directed velocity graph rather than the undirected diffusion graph — so it respects the predicted direction of differentiation rather than just topological distance.

In [ ]:
scv.tl.velocity_pseudotime(adata)
scv.pl.scatter(
    adata, color='velocity_pseudotime', cmap='gnuplot',
    save='velocity_pseudotime.png',
)

### 6b — Latent time (dynamical model)

Latent time is the per-cell time coordinate inferred *jointly with* the gene kinetics in §3. Unlike velocity_pseudotime (which is a random-walk distance), latent_time has units of (relative) time and is more directly interpretable as developmental time. Available only when dynamical mode was used.

For dentate gyrus the expected pattern: low latent time at Radial Glia-like, increasing through Neuroblast → Granule immature, peaking at Granule mature.

In [ ]:
scv.tl.latent_time(adata)
scv.pl.scatter(
    adata, color='latent_time', color_map='gnuplot',
    size=80, save='latent_time.png',
)

## 7. Top dynamical genes

### 7a — Rank top genes by likelihood and per-cluster

Genes most informative for velocity — those whose splicing kinetics fit the dynamical model best. `scv.tl.rank_dynamical_genes` ranks per cluster, picking out genes that best explain transitions in or near each cluster.

In [ ]:
scv.tl.rank_dynamical_genes(adata, groupby='clusters')
df = scv.get_df(adata, 'rank_dynamical_genes/names')
print(df.head(10).to_string())

### 7b — Phase portraits for top genes

For each gene: spliced count on x, unspliced count on y, one dot per cell. The dynamical-model fit overlays as a curve. Cells above the steady-state line are inducing the gene (unspliced building up faster than splicing can clear it); cells below are repressing it. The *position* of each cell on this loop *is* its velocity for that gene.

Reading these is the most direct way to verify that the velocity model is making biological sense — a gene known to be induced during neuron maturation should show induction in the late-trajectory clusters.

In [ ]:
top_genes = adata.var['fit_likelihood'].sort_values(ascending=False).head(6).index.tolist()
print(f'Top 6 dynamical genes by fit likelihood: {top_genes}')
scv.pl.scatter(
    adata, basis=top_genes, ncols=3,
    add_outline='fit_diff_kinetics',
    save='phase_portraits_top_genes.png',
)

### 7c — Heatmap of top genes ordered by latent time

Genes (rows) by cells (columns ordered by latent_time). Reads as a developmental cascade: which genes turn on early, which turn on late, which switch off. Cell-type bar across the top so you can see which lineage stage corresponds to each region of the heatmap.

In [ ]:
top_genes_more = adata.var['fit_likelihood'].sort_values(ascending=False).head(50).index.tolist()
scv.pl.heatmap(
    adata, var_names=top_genes_more,
    sortby='latent_time', col_color='clusters',
    n_convolve=100, save='heatmap_latent_time.png',
)

## 8. Directed PAGA

### 8a — PAGA with velocity-inferred edge directions

PAGA (Wolf et al. 2019) builds a coarse-grained graph between cell-type clusters. By default edges are undirected (symmetric connectivity). With velocity, scVelo can direct each edge — `paga.connectivities` is symmetric, `paga.transitions_confidence` is asymmetric and encodes 'cluster A is upstream of cluster B' confidence.

For dentate gyrus the expected directed edges: Radial Glia-like → Neuroblast, Neuroblast → Granule immature, Granule immature → Granule mature.

In [ ]:
adata.uns['neighbors']['distances'] = adata.obsp['distances']
adata.uns['neighbors']['connectivities'] = adata.obsp['connectivities']

scv.tl.paga(adata, groups='clusters')
df_paga = scv.get_df(adata, 'paga/transitions_confidence', precision=2).T
print('Top transitions (rows = source cluster, cols = target):')
print(df_paga.style.background_gradient(cmap='Blues').data.round(2))

### 8b — Plot directed PAGA on UMAP

Nodes = cell types (sized by cell count), edges = directed transition confidence (arrows). Overlaid on the UMAP scatter so you can see graph topology and 2D layout together.

In [ ]:
scv.pl.paga(
    adata, basis='umap', size=50, alpha=0.1,
    min_edge_width=2, node_size_scale=1.5,
    save='paga_directed.png',
)

## 9. Wrap-up

What you've now seen end-to-end on real velocyto-supported data:

1. **Stream plot (5a)** — directional flow over UMAP. The headline figure.
2. **Velocity grid (5b)** — same field as arrows for verifying local direction.
3. **Velocity confidence (5c)** — where to trust the arrows and where not to.
4. **Velocity pseudotime / latent time (6a/6b)** — orderings derived from velocity rather than diffusion.
5. **Phase portraits (7b)** — per-gene splicing dynamics that *are* the underlying velocity signal.
6. **Heatmap by latent time (7c)** — developmental gene-expression cascade.
7. **Directed PAGA (8a/8b)** — cluster-level lineage tree with arrows.

If we ever did want this for the brain organoid project, the prerequisite is running velocyto (or kb-python with `--workflow lamanno`) on Bhaduri 2020 SRA fastqs and Bhaduri 2021 NeMO-equivalent fastqs to produce per-sample loom files with spliced/unspliced layers, then merging into our existing AnnData. That is the multi-day compute step we ruled out in Session 23 — but the rest of the pipeline (filter_and_normalize, moments, recover_dynamics, velocity, velocity_graph, plots) is exactly what was just shown.